In [4]:
!pip install yt-dlp

In [5]:
import re
import time
from pathlib import Path
from urllib.parse import urlparse, parse_qs, urlunparse

import pandas as pd
import yt_dlp

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.common.exceptions import WebDriverException


In [6]:
BASE_DIR = Path("facebook pages.xlsx").resolve().parent

INPUT_EXCEL_NAME = "facebook pages.xlsx"

OUTPUT_CSV = BASE_DIR / "facebook_page_insights_output.csv"
OUTPUT_EXCEL = BASE_DIR / "facebook_page_insights_output.xlsx"
RAW_VIDEO_CSV = BASE_DIR / "raw_facebook_video_details.csv"

COOKIES_FILE = BASE_DIR / "cookies.txt"


In [7]:

# ==========================
# SCRAPER SETTINGS
# ==========================

HEADLESS = False

# If Facebook blocks public page view, login in normal Chrome and set this True.
# Usually keep False first.
USE_CHROME_USER_PROFILE = False

# Change this path only if using USE_CHROME_USER_PROFILE = True
CHROME_USER_DATA_DIR = r"C:\Users\T460\AppData\Local\Google\Chrome\User Data"
CHROME_PROFILE_DIRECTORY = "Default"

MAX_SCROLLS_PER_PAGE = 60
NO_NEW_LINK_LIMIT = 7
SCROLL_PAUSE_SECONDS = 2

# None means no limit. For testing, set 5 or 10.
MAX_VIDEOS_PER_PAGE = None

SLEEP_BETWEEN_PAGES = 3


In [8]:

# ==========================
# BASIC HELPERS
# ==========================

def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("\xa0", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def safe_int(value):
    if value is None or value == "":
        return 0

    try:
        return int(value)
    except Exception:
        pass

    try:
        return int(float(value))
    except Exception:
        return 0


def parse_count(text):
    """
    Converts:
    12K -> 12000
    1.5M -> 1500000
    2,345 -> 2345
    """

    text = clean_text(text).upper().replace(",", "")

    match = re.search(r"([\d.]+)\s*([KMB]?)", text)

    if not match:
        return ""

    number = float(match.group(1))
    suffix = match.group(2)

    if suffix == "K":
        number *= 1_000
    elif suffix == "M":
        number *= 1_000_000
    elif suffix == "B":
        number *= 1_000_000_000

    return int(number)



In [9]:


# ==========================
# EXCEL HELPERS
# ==========================

def find_input_excel():
    exact_path = BASE_DIR / INPUT_EXCEL_NAME

    if exact_path.exists():
        return exact_path

    for file in BASE_DIR.glob("*.xlsx"):
        if "facebook" in file.name.lower():
            return file

    print("Script folder:", BASE_DIR)
    print("Files in script folder:")
    for item in BASE_DIR.iterdir():
        print(" -", item.name)

    raise FileNotFoundError(
        f"Input Excel file not found. Put '{INPUT_EXCEL_NAME}' in the same folder as this script."
    )


def load_pages_from_excel(excel_path):
    all_sheets = pd.read_excel(excel_path, sheet_name=None, dtype=str)

    chosen_sheet = None
    df = None

    for sheet_name, temp_df in all_sheets.items():
        temp_df = temp_df.fillna("")
        temp_df.columns = [str(c).strip() for c in temp_df.columns]

        if "page_link" in temp_df.columns:
            chosen_sheet = sheet_name
            df = temp_df
            break

    if df is None:
        available = {sheet: list(data.columns) for sheet, data in all_sheets.items()}
        raise ValueError(
            "No sheet found with required column 'page_link'. "
            f"Available sheets and columns: {available}"
        )

    if "PAGE_ID" not in df.columns:
        df["PAGE_ID"] = ""

    if "Page_name" not in df.columns:
        df["Page_name"] = ""

    print("Excel loaded:", excel_path)
    print("Using sheet:", chosen_sheet)
    print("Rows:", len(df))

    return df



In [10]:

# ==========================
# BROWSER SETUP
# ==========================

def start_browser():
    chrome_error = None
    edge_error = None

    try:
        options = ChromeOptions()

        if HEADLESS:
            options.add_argument("--headless=new")

        options.add_argument("--start-maximized")
        options.add_argument("--disable-notifications")
        options.add_argument("--disable-popup-blocking")
        options.add_argument("--ignore-certificate-errors")
        options.add_argument("--disable-blink-features=AutomationControlled")

        if USE_CHROME_USER_PROFILE:
            options.add_argument(f"--user-data-dir={CHROME_USER_DATA_DIR}")
            options.add_argument(f"--profile-directory={CHROME_PROFILE_DIRECTORY}")

        driver = webdriver.Chrome(options=options)
        driver.set_page_load_timeout(90)
        print("Chrome browser started.")
        return driver

    except Exception as e:
        chrome_error = str(e)
        print("Chrome failed. Trying Edge...")

    try:
        options = EdgeOptions()

        if HEADLESS:
            options.add_argument("--headless=new")

        options.add_argument("--start-maximized")
        options.add_argument("--disable-notifications")
        options.add_argument("--disable-popup-blocking")
        options.add_argument("--ignore-certificate-errors")

        driver = webdriver.Edge(options=options)
        driver.set_page_load_timeout(90)
        print("Edge browser started.")
        return driver

    except Exception as e:
        edge_error = str(e)

    raise RuntimeError(
        "Browser could not start.\n"
        f"Chrome error: {chrome_error}\n"
        f"Edge error: {edge_error}\n"
        "Install/update Selenium: python -m pip install --upgrade selenium"
    )


def safe_get(driver, url):
    try:
        driver.get(url)
    except WebDriverException:
        # Sometimes page load timeout happens but DOM still loads.
        pass
    time.sleep(4)


def close_popups_if_any(driver):
    """
    Best-effort popup close. This may or may not appear depending on account/region.
    """

    possible_texts = [
        "Allow all cookies",
        "Accept all cookies",
        "Accept All",
        "Allow essential and optional cookies",
        "Not now",
        "Not Now",
        "Close",
    ]

    for text in possible_texts:
        try:
            xpath = f"//*[self::button or self::div or self::span][normalize-space()='{text}']"
            elements = driver.find_elements(By.XPATH, xpath)

            for el in elements:
                if el.is_displayed():
                    driver.execute_script("arguments[0].click();", el)
                    time.sleep(1)
                    return
        except Exception:
            pass



In [11]:

# ==========================
# FACEBOOK URL HELPERS
# ==========================

def make_facebook_videos_page_url(page_link):
    """
    Build a browser URL where video links can be discovered.
    """

    page_link = clean_text(page_link)

    if not page_link:
        return ""

    if page_link.startswith("www.facebook.com"):
        page_link = "https://" + page_link

    if page_link.startswith("facebook.com"):
        page_link = "https://" + page_link

    parsed = urlparse(page_link)

    if "facebook.com" not in parsed.netloc.lower() and "fb.com" not in parsed.netloc.lower():
        return ""

    path = parsed.path.strip("/")
    query = parse_qs(parsed.query)

    if not path:
        return ""

    if "_fb_noscript" in query:
        return ""

    # profile.php?id=123 -> profile.php?id=123&sk=videos
    if path.lower().startswith("profile.php"):
        page_id = query.get("id", [""])[0]
        if page_id:
            return f"https://www.facebook.com/profile.php?id={page_id}&sk=videos"
        return ""

    # Normal page URLs
    clean_url = urlunparse((
        parsed.scheme or "https",
        parsed.netloc,
        parsed.path,
        "",
        "",
        ""
    )).rstrip("/")

    if clean_url.lower().endswith("/videos"):
        return clean_url

    return clean_url + "/videos"


def normalize_video_url(raw_href):
    """
    Extract and normalize individual Facebook video URLs.
    """

    href = clean_text(raw_href)

    if not href:
        return ""

    if href.startswith("/"):
        href = "https://www.facebook.com" + href

    if href.startswith("www.facebook.com"):
        href = "https://" + href

    if href.startswith("facebook.com"):
        href = "https://" + href

    parsed = urlparse(href)

    if "facebook.com" not in parsed.netloc.lower() and "fb.com" not in parsed.netloc.lower():
        return ""

    path = parsed.path.strip("/")
    query = parse_qs(parsed.query)

    lower_path = path.lower()

    # watch/?v=123
    if lower_path.startswith("watch"):
        video_id = query.get("v", [""])[0]
        if video_id:
            return f"https://www.facebook.com/watch/?v={video_id}"

    # reel/123
    if lower_path.startswith("reel/"):
        parts = path.split("/")
        if len(parts) >= 2 and parts[1]:
            return f"https://www.facebook.com/reel/{parts[1]}"

    # PageName/videos/123 or videos/123
    if "/videos/" in "/" + lower_path + "/":
        parts = path.split("/")
        for i, part in enumerate(parts):
            if part.lower() == "videos" and i + 1 < len(parts):
                video_id = parts[i + 1]
                video_id = re.sub(r"[^0-9A-Za-z_-]", "", video_id)

                if video_id:
                    # Keep original page/video style
                    before = "/".join(parts[:i])
                    if before:
                        return f"https://www.facebook.com/{before}/videos/{video_id}"
                    return f"https://www.facebook.com/videos/{video_id}"

    # story.php sometimes contains story_fbid, but yt-dlp usually does not need it here.
    return ""


def video_key(video_url):
    """
    Used for de-duplication.
    """

    parsed = urlparse(video_url)
    query = parse_qs(parsed.query)

    if "v" in query:
        return "watch_" + query["v"][0]

    path = parsed.path.strip("/")
    parts = path.split("/")

    if "reel" in parts:
        i = parts.index("reel")
        if i + 1 < len(parts):
            return "reel_" + parts[i + 1]

    if "videos" in parts:
        i = parts.index("videos")
        if i + 1 < len(parts):
            return "video_" + parts[i + 1]

    return video_url



In [12]:

# ==========================
# SELENIUM EXTRACTION
# ==========================

def extract_follower_count_from_page(driver):
    try:
        text = driver.find_element(By.TAG_NAME, "body").text
    except Exception:
        return ""

    text = clean_text(text)

    patterns = [
        r"([\d,.]+\s*[KMB]?)\s+followers",
        r"([\d,.]+\s*[KMB]?)\s+people\s+follow",
        r"Followed\s+by\s+([\d,.]+\s*[KMB]?)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return parse_count(match.group(1))

    return ""


def collect_video_links(driver, page_link):
    """
    Opens Facebook page/videos tab and collects individual video URLs by scrolling.
    """

    videos_page_url = make_facebook_videos_page_url(page_link)

    if not videos_page_url:
        return [], "", "Invalid Facebook page URL"

    print("Opening videos page:", videos_page_url)

    safe_get(driver, videos_page_url)
    close_popups_if_any(driver)

    follower_count = extract_follower_count_from_page(driver)

    found = {}
    no_new_rounds = 0
    last_count = 0

    for scroll_number in range(1, MAX_SCROLLS_PER_PAGE + 1):
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[href]")

        for a in anchors:
            try:
                href = a.get_attribute("href")
            except Exception:
                continue

            video_url = normalize_video_url(href)

            if not video_url:
                continue

            key = video_key(video_url)
            found[key] = video_url

        current_count = len(found)

        print(f"Scroll {scroll_number}/{MAX_SCROLLS_PER_PAGE} | video links found: {current_count}")

        if MAX_VIDEOS_PER_PAGE is not None and current_count >= MAX_VIDEOS_PER_PAGE:
            break

        if current_count == last_count:
            no_new_rounds += 1
        else:
            no_new_rounds = 0

        if no_new_rounds >= NO_NEW_LINK_LIMIT:
            break

        last_count = current_count

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE_SECONDS)

    video_urls = list(found.values())

    if MAX_VIDEOS_PER_PAGE is not None:
        video_urls = video_urls[:MAX_VIDEOS_PER_PAGE]

    return video_urls, follower_count, ""



In [13]:

# ==========================
# YT-DLP METADATA EXTRACTION
# ==========================

class QuietLogger:
    def debug(self, msg):
        pass

    def warning(self, msg):
        pass

    def error(self, msg):
        pass


def build_yt_dlp_options():
    opts = {
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True,
        "skip_download": True,
        "socket_timeout": 30,
        "retries": 2,
        "extractor_retries": 2,
        "logger": QuietLogger(),
    }

    if COOKIES_FILE.exists():
        opts["cookiefile"] = str(COOKIES_FILE)

    return opts


def get_view_count_from_info(info):
    for key in ["view_count", "views", "play_count", "video_view_count"]:
        val = info.get(key)
        if val not in [None, "", "N/A"]:
            return safe_int(val)
    return 0


def extract_video_metadata(video_url):
    """
    Uses yt-dlp only on individual video/reel/watch URLs.
    """

    try:
        with yt_dlp.YoutubeDL(build_yt_dlp_options()) as ydl:
            info = ydl.extract_info(video_url, download=False)

            if not info:
                return {
                    "video_url": video_url,
                    "video_id": "",
                    "video_title": "",
                    "view_count": 0,
                    "upload_date": "",
                    "duration": 0,
                    "like_count": 0,
                    "comment_count": 0,
                    "yt_dlp_status": "no_data",
                    "yt_dlp_error": "yt-dlp returned no data",
                }

            info = ydl.sanitize_info(info)

        return {
            "video_url": video_url,
            "video_id": clean_text(info.get("id", "")),
            "video_title": clean_text(info.get("title", "")),
            "view_count": get_view_count_from_info(info),
            "upload_date": clean_text(info.get("upload_date", "")),
            "duration": safe_int(info.get("duration")),
            "like_count": safe_int(info.get("like_count")),
            "comment_count": safe_int(info.get("comment_count")),
            "yt_dlp_status": "success",
            "yt_dlp_error": "",
        }

    except Exception as e:
        return {
            "video_url": video_url,
            "video_id": "",
            "video_title": "",
            "view_count": 0,
            "upload_date": "",
            "duration": 0,
            "like_count": 0,
            "comment_count": 0,
            "yt_dlp_status": "failed",
            "yt_dlp_error": str(e),
        }



In [14]:

# ==========================
# PROCESS ONE PAGE
# ==========================

def process_page(driver, page_link, page_id="", page_name=""):
    result = {
        "page_link": page_link,
        "PAGE_ID": page_id,
        "Page_name": page_name,
        "page followers": "",
        "video posts_count": 0,
        "page_views": 0,
        "status": "failed",
        "error": "",
    }

    raw_video_rows = []

    video_urls, follower_count, collect_error = collect_video_links(driver, page_link)

    if follower_count != "":
        result["page followers"] = follower_count

    if collect_error:
        result["status"] = "invalid_url_skipped"
        result["error"] = collect_error
        return result, raw_video_rows

    if not video_urls:
        result["status"] = "no_video_links_found"
        result["error"] = "No video links found by Selenium. Page may need login/cookies, may have no videos, or Facebook layout blocked scraping."
        return result, raw_video_rows

    print(f"Total unique video links collected: {len(video_urls)}")
    print("Extracting video metadata using yt-dlp...")

    total_views = 0
    successful_video_metadata = 0

    for idx, video_url in enumerate(video_urls, start=1):
        print(f"  yt-dlp video {idx}/{len(video_urls)}")

        meta = extract_video_metadata(video_url)

        if meta["yt_dlp_status"] == "success":
            successful_video_metadata += 1

        total_views += safe_int(meta["view_count"])

        raw_video_rows.append({
            "page_link": page_link,
            "PAGE_ID": page_id,
            "Page_name": page_name,
            **meta,
        })

    result["video posts_count"] = len(video_urls)
    result["page_views"] = total_views

    if successful_video_metadata == 0:
        result["status"] = "video_links_found_but_yt_dlp_failed"
        result["error"] = "Selenium found video links, but yt-dlp could not extract metadata. Try cookies.txt or login browser profile."
    elif result["page followers"] == "":
        result["status"] = "success_followers_not_available"
    else:
        result["status"] = "success"

    return result, raw_video_rows



In [15]:

# ==========================
# MAIN
# ==========================

def main():
    print("Script folder:", BASE_DIR)

    excel_path = find_input_excel()
    df = load_pages_from_excel(excel_path)

    final_rows = []
    all_video_rows = []

    driver = start_browser()

    try:
        total_rows = len(df)

        for index, row in df.iterrows():
            page_link = clean_text(row.get("page_link", ""))
            page_id = clean_text(row.get("PAGE_ID", ""))
            page_name = clean_text(row.get("Page_name", ""))

            print("\n" + "=" * 70)
            print(f"Processing {index + 1}/{total_rows}: {page_name}")
            print("Page link:", page_link)

            if not page_link:
                result = {
                    "page_link": page_link,
                    "PAGE_ID": page_id,
                    "Page_name": page_name,
                    "page followers": "",
                    "video posts_count": 0,
                    "page_views": 0,
                    "status": "empty_url_skipped",
                    "error": "Empty page_link",
                }
                raw_videos = []
            else:
                result, raw_videos = process_page(
                    driver=driver,
                    page_link=page_link,
                    page_id=page_id,
                    page_name=page_name,
                )

            final_rows.append(result)
            all_video_rows.extend(raw_videos)

            # Save after every page. Original Excel is not modified.
            pd.DataFrame(final_rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
            pd.DataFrame(final_rows).to_excel(OUTPUT_EXCEL, index=False)

            if all_video_rows:
                pd.DataFrame(all_video_rows).to_csv(RAW_VIDEO_CSV, index=False, encoding="utf-8-sig")

            print("Status:", result["status"])
            print("Followers:", result["page followers"])
            print("Videos:", result["video posts_count"])
            print("Total Views:", result["page_views"])
            if result["error"]:
                print("Note:", result["error"])

            time.sleep(SLEEP_BETWEEN_PAGES)

    finally:
        driver.quit()

    print("\nDONE")
    print("Original Excel file was not modified.")
    print("Output CSV:", OUTPUT_CSV)
    print("Output Excel:", OUTPUT_EXCEL)
    print("Raw video details:", RAW_VIDEO_CSV)


if __name__ == "__main__":
    main()


Script folder: C:\Users\Muhammad Sheraz\Desktop\sheraz(intern_task)\facebook page\main.ipynb
Excel loaded: C:\Users\Muhammad Sheraz\Desktop\sheraz(intern_task)\facebook page\main.ipynb\facebook pages.xlsx
Using sheet: Sheet2
Rows: 44
Chrome browser started.

Processing 1/44: Fruity AI
Page link: https://www.facebook.com/profile.php?id=1040451862493310
Opening videos page: https://www.facebook.com/profile.php?id=1040451862493310&sk=videos


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=148.0.7778.179)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7645c7de5+14895]
	chromedriver!GetHandleVerifier [0x7ff7645c7e50+14900]
	chromedriver!(No symbol) [0x7ff76432d5ad]
	chromedriver!(No symbol) [0x7ff7643040f2]
	chromedriver!(No symbol) [0x7ff7643b6f56]
	chromedriver!(No symbol) [0x7ff7643d4172]
	chromedriver!(No symbol) [0x7ff764379df8]
	chromedriver!(No symbol) [0x7ff76437ace3]
	chromedriver!GetHandleVerifier [0x7ff7648dcc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff7648d7375+323e25]
	chromedriver!GetHandleVerifier [0x7ff7648fbc82+348732]
	chromedriver!GetHandleVerifier [0x7ff7645e6045+32af5]
	chromedriver!GetHandleVerifier [0x7ff7645eecec+3b79c]
	chromedriver!GetHandleVerifier [0x7ff7645d1bc4+1e674]
	chromedriver!GetHandleVerifier [0x7ff7645d1d54+1e804]
	chromedriver!GetHandleVerifier [0x7ff7645b60e7+2b97]
	KERNEL32!BaseThreadIn

In [2]:
pip install openpyxl --user

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
